# Lab : POS Tagging

## Objectives:

Use a HMM model (in a *supervised* manner) and a CRF model for a task of POS-Tagging
1. Look at the data: understand the structure, the labels, collect some statistics. 
2. Implement a HMM model for **supervised learning**:
    - From scratch: we will need to implement a ```fit``` method. What does MLE gives for our parameters ?
    - We will need to use the Viterbi algorithm for our ```predict``` method
3. Perform classification:
    - Compute accuracy
    - Look at the confusion matrix between tags
4. Use an existing CRF implementation, with handmade features
    - Perform classification and look at the confusion matrix
5. Use an existing HMM implementation, made for **unsupervised learning**
    - Apply it: does it work ?
    - Look at the transition probabilities that are learnt

## Necessary dependencies

We will need the following packages:
- The Natural Language Toolkit : http://www.nltk.org/install.html for our data
- The Machine Learning API Scikit-learn : http://scikit-learn.org/stable/install.html for our methods - our models will be based on the same structure as ```sklearn```, using the ```hmmlearn``` and ```sklearn_crfsuite``` library

In [ ]:
import nltk
import math
import numpy as np
# For visualization
import matplotlib.pyplot as plt
import seaborn as sns
# For counting 
from collections import defaultdict, Counter 

In [ ]:
# Models we will use 
from hmmlearn import hmm
import sklearn_crfsuite

In [ ]:
# For data 
from sklearn.model_selection import train_test_split

The Penn Treebank is a widely used annotated corpus of English, created at the University of Pennsylvania. It contains data from sources such as newspapers, with each sentence annotated for part-of-speech (POS) tags and syntactic structure - it is standard for tasks such as POS tagging and syntactic parsing. 
The ```NLTK``` package contains a subset of 10% of this dataset, in ```treebank```:

In [ ]:
nltk.download('treebank')
corpus = nltk.corpus.treebank.tagged_sents()
print(corpus[0])

In [ ]:
# Split train/test
train_sents, test_sets = train_test_split(corpus, shuffle = False)

### I - Exploring data

Explore the *training data*: 
- Count the number of tags and visualize their frequency,
- Look at the rank / frequency curve of words, in the log-space, to visualize *Zipf's distribution*
- Display an histogram of sentence lengths to have an idea of how the sequences are distributed 
<div class='alert alert-block alert-info'>
            Code:</div>

## II - Implementing a supervised HMM

We will now implement a ```HMMTagger``` class. It contains:
- A ```fit``` method, which takes as input sentences and their tags and learn HMM parameters: transition probabilities $A$, emission probabilities $B$, initial state probabilities $\pi$
    - What should be the values of these parameters, depending of the frequencies of *tag-tag* transitions and *tag-word* emission observed in data ?
        - We can compute *empirical probabilities*, which would be what we would get if actually computing Maximum Likelihood Estimation
        - This is exactly like learning the parameters of the *Naive Bayes* in the first class
    - We will need to use *smoothing* with $\alpha$ to avoid having a probabilities at $0$.
- A ```predict``` method which takes as input a sequence of words and outputs the most probable sequence of tags given the parameters.
    - It should implement the Viterbi algorithm, including the recursion followed by backtracking.
    - The practical difficulty is to deal with possibly *unknown* words: in this case, we arbitrarily use the emission probability $\frac{1}{|V|}$   
<div class='alert alert-block alert-info'>
            Code:</div>

In [ ]:
class HMMTagger:
    def __init__(self, alpha=1.0):
        self.vocab = set()
        self.tags = set()
        # For Laplace smoothing
        self.alpha = alpha

    # Supervised training: we need to estimate the parameters given the training data
    def fit(self, tagged_sentences):
        # We don't know yet how many tags/words we will encounter
        # We use defaultdict(Counter) to be able to add them along the way
        transition_counts = defaultdict(Counter)
        emission_counts = defaultdict(Counter)
        # We'll need this for initial probabilities
        tag_counts = Counter()

        # Getting counts on all data
        for sent in tagged_sentences:
            prev_tag = "<START>"   
            for word, tag in sent:
                self.vocab.add(word)
                self.tags.add(tag)
                # Count transitions of 'prev_tag' to 'tag'
                ...
                # Count emissions of 'word' for 'tag'
                ...
                # Count tags
                ...
                prev_tag = tag  
            # End transition
            transition_counts[prev_tag]["<END>"] += 1

        # Necessary to get parameters: vocabulary of tags and words
        self.tag_to_idx = {t:i for i,t in enumerate(self.tags)}
        self.idx_to_tag = {i:t for i,t in enumerate(self.tags)}
        self.word_to_idx = {w:i for i,w in enumerate(self.vocab)}
        K = len(self.tags)
        V = len(self.vocab)

        # Transition probabilities
        self.A = ...
        for prev_tag in self.tags:
            ...
            for curr_tag in self.tags:
                ...
        
        # Initial state probabilities
        self.pi = ...
        for tag in self.tags:
            ...

        # Emission probabilities
        self.B = ...        
        for tag in self.tags:
            ...
            for word in self.vocab:
                ...
                
    # Prediction: with Viterbi
    def predict(self, sentence):
        K = len(self.tags)
        T = len(sentence)
        # Map words to indexes
        obs = []
        for w in sentence:
            if w in self.word_to_idx:
                ...
            else:
                obs.append(-1)  # Unknown word
        
        # Creating the necessary tables
        delta = ...
        psi = ...
        
        # Initialization
        if obs[0] == -1:
            # In case of unknown word, we use a uniform probability
            delta[0] = ... + np.log(1.0 / len(self.vocab))
        else:
            delta[0] = ...
        
        # Recursion
        for t in range(1, T):
            scores = ... 
            if obs[t] == -1: # Unknown word
                delta[t] = ... + np.log(1.0 / len(self.vocab))
            else:
                delta[t] = ...
            ...
        
        # Backtrack
        tags_idx = []
        last = ... # Best last state
        tags_idx.append(last) 
        for t in reversed(range(1, T)):
            ...
        tags_idx.reverse()
        
        return [self.idx_to_tag[i] for i in tags_idx]

In [ ]:
# Train
tagger = HMMTagger()
tagger.fit(train_sents)

In [ ]:
# Test on one sentence
words = [w for w, t in test_sents[0]]
true_tags = [t for w, t in test_sents[0]]

pred_tags = tagger.predict(words)
print(list(zip(words, pred_tags)))

## III - Evaluation

Complete the following functions, to compute accuracy and visualize the confusion matrix. 

<div class='alert alert-block alert-info'>
            Code:</div>

In [ ]:
def pos_accuracy(y_true, y_pred):
    correct = 0
    total = 0
    ...
    ...
    return correct / total

In [ ]:
...

In [ ]:
...

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_confusion(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(15,12))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("POS Tag Confusion Matrix")
    plt.show()

In [ ]:
...

Look at the confusion matrix for the test data. Give the most common errors made by the model. Check the tag documentation to try to understand where they might come from. 

<div class='alert alert-block alert-warning'>
            Question:</div>

## IV - Using a CRF

In this part, we will use a ```CRF``` model from the ```sklearn_crfsuite``` package. 
The CRF is a supervised model, which uses *local features* that are aggregated through Viterbi when we want to do a prediction.
Learning is made through parameter updates - however, computing those gradients would require implementing the baum-welch algorithm ! So, we will stick to an existing implementation. 

We will use the following ```word_features``` function, which will give us a dictionary of features for each position in the sentence. 

In [ ]:
def word_features(sentence, i):
    word = sentence[i][0]
    features = {
        'word': word,
        'is_first': i == 0,
        'is_last': i == len(sentence) - 1,
        'is_capitalized': word[0].upper() == word[0],
        'is_all_caps': word.upper() == word,
        'is_all_lower': word.lower() == word,
        'prefix-1': word[0],
        'prefix-2': word[:2],
        'prefix-3': word[:3],
        'suffix-1': word[-1],
        'suffix-2': word[-2:],
        'suffix-3': word[-3:],
        'prev_word': '' if i == 0 else sentence[i-1][0],
        'next_word': '' if i == len(sentence)-1 else sentence[i+1][0],
        'has_hyphen': '-' in word,
        'is_numeric': word.isdigit(),
        'capitals_inside': word[1:].lower() != word[1:]
    }
    return features

We need then to convert data at the right format to train the mode, do prediction, and compute accuracy. 
Complete the following: 

<div class='alert alert-block alert-info'>
            Code:</div>

In [ ]:
def data_crf(sentences): 
    X = []
    y = []
    ...
    return X, y

In [ ]:
X_train, y_train = data_crf(train_sents)
X_test, y_test = data_crf(test_sents)

In [ ]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crf.fit(X_train, y_train)
y_pred = crf.predict(X_test)

Compare the results with our HMM model, in terms of accuracy and confusion matrix. Try to explain them using your knowledge of the models. 

<div class='alert alert-block alert-warning'>
            Question:</div>

## V - Using a HMM in an unsupervised manner

In the last part of this lab, we may want to use an existing HMM library: ```hmmlearner`̀ `
Here is the documentation: https://hmmlearn.readthedocs.io/en/latest/index.html

Looking at the proposed model, we can see there is no **supervised learning option in this library**. 
Still, we can test to train the model in an *unsupervised manner*. Your task here is:
- To pick the right ```hmmlearner``` model - which one, in the documentation, is better suited to our task ?
- To transform the data into a format adapted to this model - you will need to implement your own vocabulary !
- Compute the accuracy. Were those results expected ?
- Visualize the transition probability matrices of the first HMM trained, and this one. Again, try to look at the high values in the matrux and check if they make sense.

<div class='alert alert-block alert-info'>
            Code:</div>
            
<div class='alert alert-block alert-warning'>
            Question:</div>

In [ ]:
def plot_hmm_matrix(matrix, tags):
    plt.figure(figsize=(10,8))
    sns.heatmap(matrix, xticklabels=tags, yticklabels=tags, cmap="viridis", annot=False)
    plt.xlabel("To tag")
    plt.ylabel("From tag")
    plt.title("Transition Matrix")
    plt.show()

In [ ]:
# For a model in hmmlearner, you find the transition matrix as model.transmat_